# 06. 음성 검색 시연 (STT + TTS)

마이크 버튼 → 말하기 → 문장 인식(STT) → 출발지, 목적지, 출발 시각 해석 → 추천 5곳 + TMAP 지도 → 결과 읽어 주기(TTS).

| 단계 | 어디서 | 비용 |
|---|---|---|
| 음성 인식 (STT) | 브라우저 내장 (Web Speech API, 한국어) | 무료 |
| 문장 해석, 추천 | 내 PC에서 띄운 예측 서버 (`/voice`) | TMAP 한도 안 |
| 읽어 주기 (TTS) | 브라우저 내장 (speechSynthesis) | 무료 |

크롬 또는 엣지에서 열어야 한다. 처음 열면 마이크, 위치 권한을 묻는다 (둘 다 허용).

## 1. 서버 라이브러리 확인 (처음 한 번)

없으면 설치한다. 설치했다면 Kernel → Restart Kernel 후 다음 셀부터.

In [ ]:
try:
    import fastapi, uvicorn
    print("fastapi", fastapi.__version__, "/ uvicorn", uvicorn.__version__, "준비됨")
except ImportError:
    %pip install fastapi uvicorn

## 2. 예측 서버 켜고 시연 페이지 열기

모델과 최신 주차 기록을 불러오느라 10~30초 걸린다. 브라우저에 시연 페이지가 새 탭으로 열린다.

In [ ]:
import os, sys, time, subprocess, webbrowser, urllib.request
from pathlib import Path

ML = Path("..").resolve()
URL = "http://localhost:8000"

def alive():
    try:
        return urllib.request.urlopen(URL + "/health", timeout=2).status == 200
    except Exception:
        return False

if not alive():
    env = {**os.environ, "LOKY_MAX_CPU_COUNT": "4"}
    log = open(ML / "server.log", "w", encoding="utf-8")
    server = subprocess.Popen([sys.executable, "-m", "uvicorn", "api.app:app", "--port", "8000"],
                              cwd=ML, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(90):
        if alive() or server.poll() is not None:
            break
        time.sleep(1)
print("서버:", "켜짐" if alive() else "실패 (아래 로그 확인)")
if alive():
    webbrowser.open(URL + "/demo")
    print("시연 페이지:", URL + "/demo")
else:
    print((ML / "server.log").read_text(encoding="utf-8")[-2000:])

## 3. 문장 해석만 시험하기 (TMAP 호출 없음)

말할 문장을 직접 입력해서 출발지, 목적지, 출발 시각이 제대로 나뉘는지 본다.

In [ ]:
sys.path.insert(0, str(ML))
from jariml.voice import parse_voice
q = parse_voice(input("문장: ") or "수원 SK스카이뷰에서 교대역까지 내일 아침 8시 반에 출발")
print("출발지:", q["origin"] or "현재 위치", "/ 목적지:", q["dest"], "/ 출발:", q["depart"] or "지금")

## 4. 서버 끄기

시연이 끝나면 실행한다. 노트북 커널을 재시작해도 서버는 꺼진다.

In [ ]:
try:
    server.terminate(); print("서버 꺼짐")
except NameError:
    print("이 노트북에서 켠 서버가 없다")